In [ ]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [145]:
# Дефиниране на GNN енкодер
from torch_geometric.nn import DeepGraphInfomax
from torch_geometric.nn import GCNConv

class Encoder(torch.nn.Module):

    def __init__(self,
                 in_channels,
                 hidden_channels):

        super().__init__()

        self.conv = GCNConv(
            in_channels,
            hidden_channels)

    def forward(self,
                x,
                edge_index):

        x = self.conv(
            x,
            edge_index)

        return x.relu()

In [146]:
# Създаване на DGI модела
model = DeepGraphInfomax(
    hidden_channels=512,
    encoder=Encoder(
        dataset.num_features,
        512),
    summary=lambda z, *args, **kwargs:
        torch.sigmoid(z.mean(dim=0)),
    corruption=lambda x, edge_index:
        (x[torch.randperm(x.size(0))],
         edge_index))

In [147]:
# Инициализиране на оптимизатора
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001)

In [148]:
# Обучение на модела
model.train()
for epoch in range(200):

    optimizer.zero_grad()

    pos_z, neg_z, summary = model(
        data.x,
        data.edge_index)

    loss = model.loss(
        pos_z,
        neg_z,
        summary)

    loss.backward()

    optimizer.step()

    if epoch % 20 == 0:
        print(
            f"Epoch: {epoch:03d}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch: 000, Loss: 1.4305
Epoch: 020, Loss: 1.3556
Epoch: 040, Loss: 1.2179
Epoch: 060, Loss: 0.8246
Epoch: 080, Loss: 0.3345
Epoch: 100, Loss: 0.1609
Epoch: 120, Loss: 0.1175
Epoch: 140, Loss: 0.0800
Epoch: 160, Loss: 0.0561
Epoch: 180, Loss: 0.0523


In [149]:
# Извличане на представянията
model.eval()

with torch.no_grad():

    embeddings = model.encoder(
        data.x,
        data.edge_index)

print(embeddings.shape)

torch.Size([2708, 512])


In [150]:
# Класифициране на възлите
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)

clf.fit(embeddings[data.train_mask].cpu(),
    data.y[data.train_mask].cpu())

pred = clf.predict(embeddings[data.test_mask].cpu())

In [151]:
# Оценяване на модела
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

accuracy = accuracy_score(
    data.y[data.test_mask].cpu(),
    pred
)

f1 = f1_score(
    data.y[data.test_mask].cpu(),
    pred,
    average="macro")

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1-score: {f1:.4f}")

Accuracy: 0.8010
Macro F1-score: 0.7849
